<a href="https://colab.research.google.com/github/Alind755/CommentExtracter/blob/master/Qwen_Image_2_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Qwen-Image-2.1 Text-to-Image Generation (with LORa Support)
### 🔴 **Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)**

This notebook runs the official Qwen-Image-2.1 INT8 model with support for:
- Native resolution selector (up to 2048x2048 2K)
- Text encoder (Qwen3-VL 8B)
- Dynamic LoRA injection with sequential offloading
- In-memory caching (`QwenImage21Cache`)

In [1]:
# @title 1. Environment Setup & Model Synchronization
import os
import subprocess
import time
from IPython.display import display, HTML

t0 = time.time()

# 1. Package Installation via uv
!pip install -q uv
!apt-get update -qq && apt-get install -y -qq aria2

if not os.path.exists("/content/engine"):
    !git clone -q https://github.com/comfyanonymous/ComfyUI.git /content/engine
%cd /content/engine

!uv pip install --system torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!uv pip install --system -r requirements.txt
!uv pip install --system websocket-client requests Pillow gdown

# 2. Foolproof Download Engine
def foolproof_download(url: str, output_dir: str, exact_filename: str, min_size_mb: float = 100.0):
    os.makedirs(output_dir, exist_ok=True)
    target_file = os.path.join(output_dir, exact_filename)
    aria_meta_file = target_file + ".aria2"

    if os.path.exists(target_file) and not os.path.exists(aria_meta_file):
        size_mb = os.path.getsize(target_file) / (1024 * 1024)
        if size_mb >= min_size_mb:
            return "verified", size_mb

    cmd = [
        "aria2c",
        "--console-log-level=error",
        "-c",
        "-x", "16",
        "-s", "16",
        "-k", "1M",
        "--auto-file-renaming=false",
        "--allow-overwrite=true",
        "--content-disposition=false",
        "-d", output_dir,
        "-o", exact_filename,
        url
    ]

    result = subprocess.run(cmd)
    if result.returncode != 0 or not os.path.exists(target_file):
        raise RuntimeError(f"Download failed for {exact_filename}")

    if os.path.exists(aria_meta_file):
        os.remove(aria_meta_file)

    size_mb = os.path.getsize(target_file) / (1024 * 1024)
    return "downloaded", size_mb

# Base Model Registry (Only Essential Core Weights)
models_manifest = [
    {
        "name": "Diffusion Model (INT8 convrot)",
        "file": "qwen_image_2.1_int8_convrot.safetensors",
        "dir": "/content/engine/models/diffusion_models",
        "url": "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/diffusion_models/qwen_image_2.1_int8_convrot.safetensors",
        "min_mb": 500.0
    },
    {
        "name": "Text Encoder (Qwen3-VL 8B INT8)",
        "file": "qwen3vl_8b_int8_convrot.safetensors",
        "dir": "/content/engine/models/text_encoders",
        "url": "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/text_encoders/qwen3vl_8b_int8_convrot.safetensors",
        "min_mb": 500.0
    },
    {
        "name": "VAE (BF16)",
        "file": "qwen_image_2.1_vae_bf16.safetensors",
        "dir": "/content/engine/models/vae",
        "url": "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/vae/qwen_image_2.1_vae_bf16.safetensors",
        "min_mb": 100.0
    }
]

!mkdir -p /content/engine/models/diffusion_models /content/engine/models/text_encoders /content/engine/models/vae /content/engine/models/loras

table_rows = ""
for item in models_manifest:
    status, size = foolproof_download(item["url"], item["dir"], item["file"], item["min_mb"])
    badge_color = "#3fb950" if status == "verified" else "#58a6ff"
    badge_text = "VERIFIED" if status == "verified" else "SYNCED"
    table_rows += f"""
    <tr style="border-bottom: 1px solid #21262d;">
      <td style="padding: 8px 12px; color: #c9d1d9;">{item['name']}</td>
      <td style="padding: 8px 12px; color: #8b949e; font-family: monospace;">{item['file']}</td>
      <td style="padding: 8px 12px; color: #8b949e; text-align: right;">{size:.1f} MB</td>
      <td style="padding: 8px 12px; text-align: center;">
        <span style="background: rgba(46, 160, 67, 0.15); color: {badge_color}; border: 1px solid {badge_color}40; padding: 2px 8px; border-radius: 12px; font-size: 10px; font-weight: 600;">{badge_text}</span>
      </td>
    </tr>
    """

total_elapsed = time.time() - t0
display(HTML(f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; background: #0d1117; border: 1px solid #30363d; border-radius: 10px; padding: 18px; max-width: 820px; box-shadow: 0 8px 24px rgba(0,0,0,0.3); margin: 10px 0;">
  <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid #21262d; padding-bottom: 12px; margin-bottom: 12px;">
    <div style="display: flex; align-items: center; gap: 8px;">
      <span style="width: 10px; height: 10px; border-radius: 50%; background: #3fb950; box-shadow: 0 0 10px #238636;"></span>
      <span style="color: #f0f6fc; font-weight: 600; font-size: 13px;">Qwen-Image-2.1 Base Weights & Registry</span>
    </div>
    <span style="background: rgba(56, 139, 253, 0.15); color: #58a6ff; border: 1px solid rgba(56, 139, 253, 0.4); padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: 600;">
      READY ({total_elapsed:.1f}s)
    </span>
  </div>

  <table style="width: 100%; border-collapse: collapse; font-size: 12px;">
    <thead>
      <tr style="border-bottom: 1px solid #30363d; color: #8b949e; text-align: left;">
        <th style="padding: 6px 12px;">Module</th>
        <th style="padding: 6px 12px;">Target Filename</th>
        <th style="padding: 6px 12px; text-align: right;">Size</th>
        <th style="padding: 6px 12px; text-align: center;">Status</th>
      </tr>
    </thead>
    <tbody>
      {table_rows}
    </tbody>
  </table>
</div>
"""))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.6/20.6 MB 26.4 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libcares2:amd64.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../libcares2_1.27.0-1.0ubuntu1_amd64.deb ...
Unpacking libcares2:amd64 (1.27.0-1.0ubuntu1) ...
Selecting previously unselected package libaria2-0:amd64.
Preparing to unpack .../libaria2-0_1.37.0+debian-1build3_amd64.deb ...
Unpacking libaria2-0:amd64 (1.37.0+debian-1build3) ...
Selecting previously unselected package aria2.
Preparing to unpack .../aria2_1.37.0+debian-1build3_amd64.deb ...
Unpacking aria2 (1.37.0+debian-1build3) ...
Setting up libcares2:amd64 (1.27.0-1.0ubuntu1) ...
Setting up libaria2-0:amd64 (1.37.0+debian-1build3) ...
Setting up aria2 (1.37.0+debian-1build

Module,Target Filename,Size,Status
Diffusion Model (INT8 convrot),qwen_image_2.1_int8_convrot.safetensors,6920.6 MB,SYNCED
Text Encoder (Qwen3-VL 8B INT8),qwen3vl_8b_int8_convrot.safetensors,8917.6 MB,SYNCED
VAE (BF16),qwen_image_2.1_vae_bf16.safetensors,644.2 MB,SYNCED


In [ ]:
# @title 2. (Optional) Multi-Source LoRA Downloader & Registry
import os
import re
import subprocess
import requests
from IPython.display import display, HTML

lora_source_url = "" # @param {type:"string"}
lora_filename = "custom_lora.safetensors" # @param {type:"string"}
civitai_token = "" # @param {type:"string"}
hf_token = "" # @param {type:"string"}

def download_lora_multisource(url: str, output_dir: str, exact_filename: str, civ_token: str = "", hf_tok: str = ""):
    os.makedirs(output_dir, exist_ok=True)
    target_file = os.path.join(output_dir, exact_filename)
    aria_meta_file = target_file + ".aria2"

    if os.path.exists(target_file) and not os.path.exists(aria_meta_file):
        size_mb = os.path.getsize(target_file) / (1024 * 1024)
        if size_mb >= 1.0:
            return "Existing / Verified", size_mb

    clean_url = url.strip()
    platform = "Direct Link"

    if "drive.google.com" in clean_url or "docs.google.com" in clean_url:
        platform = "Google Drive"
        import gdown
        gdown.download(clean_url, target_file, fuzzy=True, quiet=False)

    elif "civitai.com" in clean_url:
        platform = "Civitai"
        m_ver = re.search(r'modelVersionId=(\d+)', clean_url)
        if m_ver:
            clean_url = f"https://civitai.com/api/download/models/{m_ver.group(1)}"
        elif "/models/" in clean_url and "/api/" not in clean_url:
            m_id = re.search(r'/models/(\d+)', clean_url)
            if m_id:
                try:
                    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
                    if civ_token.strip():
                        headers["Authorization"] = f"Bearer {civ_token.strip()}"
                    res = requests.get(f"https://civitai.com/api/v1/models/{m_id.group(1)}", headers=headers, timeout=6).json()
                    ver_id = res["modelVersions"][0]["id"]
                    clean_url = f"https://civitai.com/api/download/models/{ver_id}"
                except Exception as e:
                    print(f"Notice: Civitai auto-resolution defaulted to URL: {e}")

        header_flags = ["--header=User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64)"]
        if civ_token.strip():
            header_flags.append(f"--header=Authorization: Bearer {civ_token.strip()}")
            delimiter = "&" if "?" in clean_url else "?"
            clean_url = f"{clean_url}{delimiter}token={civ_token.strip()}"

        cmd = [
            "aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16", "-k", "1M",
            "--auto-file-renaming=false", "--allow-overwrite=true", "--content-disposition=false",
            "-d", output_dir, "-o", exact_filename, *header_flags, clean_url
        ]
        subprocess.run(cmd)

    elif "huggingface.co" in clean_url:
        platform = "Hugging Face"
        clean_url = clean_url.replace("/blob/", "/resolve/")
        header_flags = []
        if hf_tok.strip():
            header_flags.append(f"--header=Authorization: Bearer {hf_tok.strip()}")

        cmd = [
            "aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16", "-k", "1M",
            "--auto-file-renaming=false", "--allow-overwrite=true", "--content-disposition=false",
            "-d", output_dir, "-o", exact_filename, *header_flags, clean_url
        ]
        subprocess.run(cmd)

    else:
        cmd = [
            "aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16", "-k", "1M",
            "--auto-file-renaming=false", "--allow-overwrite=true", "--content-disposition=false",
            "-d", output_dir, "-o", exact_filename, clean_url
        ]
        subprocess.run(cmd)

    if not os.path.exists(target_file) or os.path.getsize(target_file) < 1024 * 1024:
        raise RuntimeError(f"Failed to download LoRA from {platform}. Please check the URL, token, or file permissions.")

    if os.path.exists(aria_meta_file):
        os.remove(aria_meta_file)

    size_mb = os.path.getsize(target_file) / (1024 * 1024)
    return platform, size_mb

if lora_source_url.strip():
    platform_name, size = download_lora_multisource(
        url=lora_source_url.strip(),
        output_dir="/content/engine/models/loras",
        exact_filename=lora_filename.strip(),
        civ_token=civitai_token,
        hf_tok=hf_token
    )
    display(HTML(f"""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; background: #0d1117; border: 1px solid #30363d; border-left: 4px solid #a371f7; border-radius: 8px; padding: 12px 16px; max-width: 760px; margin-top: 8px;">
      <div style="display: flex; justify-content: space-between; align-items: center;">
        <span style="color: #f0f6fc; font-weight: 600; font-size: 13px;">📦 Custom LoRA Acquired</span>
        <span style="background: rgba(163, 113, 247, 0.15); color: #d2a8ff; border: 1px solid rgba(163, 113, 247, 0.4); padding: 2px 8px; border-radius: 12px; font-size: 10px; font-weight: 600;">{platform_name}</span>
      </div>
      <div style="color: #8b949e; font-size: 12px; margin-top: 6px; font-family: monospace;">
        Target File: <span style="color: #d2a8ff;">models/loras/{lora_filename.strip()}</span> ({size:.1f} MB)
      </div>
    </div>
    """))
else:
    lora_dir = "/content/engine/models/loras"
    available_loras = os.listdir(lora_dir) if os.path.exists(lora_dir) else []
    loras_html = ", ".join(available_loras) if available_loras else "None loaded"
    display(HTML(f"""
    <div style="font-family: monospace; color: #8b949e; font-size: 11px; padding: 6px;">
      ⓘ No new LoRA URL entered. Available local LoRAs: <span style="color: #79c0ff;">{loras_html}</span>
    </div>
    """))

In [2]:
# @title 3. Start Inference Server
import time
import subprocess
import requests
import os
from IPython.display import display, HTML

# Host memory conservation flags
os.environ["TORCH_ALLOW_TF32_CUBLAS_OVERRIDE"] = "1"
os.environ["CUDA_MODULE_LOADING"] = "LAZY"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def launch_inference_server():
    global server_proc
    server_proc = subprocess.Popen(
        [
            "python", "main.py",
            "--listen", "127.0.0.1",
            "--port", "8188",
            "--disable-pinned-memory",
            "--mmap-torch-files",
            "--fast",
            "--cuda-malloc",
            "--fp16-vae",
            "--preview-method", "none"
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

launch_inference_server()

server_ready = False
t_poll = time.time()
while time.time() - t_poll < 60:
    try:
        r = requests.get("http://127.0.0.1:8188/system_stats", timeout=1)
        if r.status_code == 200:
            server_ready = True
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(1.2)

if server_ready:
    display(HTML(f"""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; background: #0d1117; border: 1px solid #30363d; border-radius: 8px; padding: 12px 16px; max-width: 760px; display: flex; align-items: center; justify-content: space-between;">
      <div style="display: flex; align-items: center; gap: 10px;">
        <span style="display: inline-block; width: 10px; height: 10px; border-radius: 50%; background: #3fb950; box-shadow: 0 0 10px #238636;"></span>
        <span style="color: #f0f6fc; font-weight: 600; font-size: 13px;">Inference Backend Server (Protected against OOM)</span>
        <span style="color: #8b949e; font-size: 11px;">127.0.0.1:8188</span>
      </div>
      <span style="background: rgba(56, 139, 253, 0.15); color: #58a6ff; border: 1px solid rgba(56, 139, 253, 0.4); padding: 2px 10px; border-radius: 12px; font-size: 11px; font-weight: 600;">
        ONLINE ({time.time() - t_poll:.1f}s)
      </span>
    </div>
    """))
else:
    display(HTML("""
    <div style="background: #ff000022; border: 1px solid #ff0000; color: #ff7b72; padding: 12px; border-radius: 6px;">
      ✕ Backend server failed to start within timeout.
    </div>
    """))

In [ ]:
from pathlib import Path
# @title 4. Image Generation
import json
import uuid
import urllib.request
import urllib.parse
import websocket
from PIL import Image
import io
import time
import random
import math
import requests
from IPython.display import display, HTML

# --- Prompt Parameters ---
prompt_text = "This is an explicit photograph depicting a sexual scene involving multiple individuals. The central figure is a a   voluptuous   Indian  Actress  with glistening skin in her late 40s or early 50s  revealing    huge cleavage with hijabi pulled back hair, lying on her stomach on a white surface, possibly a bed or couch. She appears to be in a state of arousal, with her eyes closed and mouth slightly open. Her body is partially nude, revealing her buttocks and the back of her thighs.    Surrounding her are several men, all with darker skin tones. One man is positioned behind her, engaging in sexual intercourse from the rear. Another man is to her left, holding a dildo against her genital area. A third man is visible on the right side of the image, his hand resting on her buttocks.    The setting appears to be indoors, with natural light coming through a window or glass door in the background. There are potted plants and some furniture partially visible, suggesting a domestic environment. The overall atmosphere is intimate and erotic, focusing on the sexual activity between the individuals." # @param {type:"string"}
negative_prompt = "blurry, low quality, artifacts, distorted, out of focus" # @param {type:"string"}

# --- Dimension Settings (Custom Width & Height up to 2048x2048) ---
dimension_mode = "Custom" # @param ["Custom", "Preset Aspect Ratio"]

# Active when dimension_mode is "Custom"
custom_width = 1920 # @param {type:"integer"}
custom_height = 1080 # @param {type:"integer"}

# Active when dimension_mode is "Preset Aspect Ratio"
aspect_ratio = "16:9 (Widescreen)" # @param ["1:1 (Square)", "16:9 (Widescreen)", "9:16 (Portrait Widescreen)", "21:9 (Ultrawide)", "9:21 (Tall Ultrawide)", "4:3 (Standard)", "3:4 (Portrait Standard)", "3:2 (Photo)", "2:3 (Portrait Photo)", "5:4 (Landscape Tall)", "4:5 (Portrait Tall)", "1:2 (Tall Vertical)", "2:1 (Panoramic)"]
megapixels = 1.0 # @param {type:"slider", min:0.5, max:4.2, step:0.1}

# --- Sampling Parameters ---
steps = 18 # @param {type:"integer"}
seed = -1 # @param {type:"integer"}

# --- LoRA Controls ---
use_lora = False # @param {type:"boolean"}
lora_name = "custom_lora.safetensors" # @param {type:"string"}
lora_model_strength = 1.0 # @param {type:"slider", min:0.0, max:2.0, step:0.05}
lora_clip_strength = 1.0 # @param {type:"slider", min:0.0, max:2.0, step:0.05}

if seed == -1:
    seed = random.randint(1, 10**14)

# --- Calculate Dimensions & Enforce 8px Alignment ---
if dimension_mode == "Custom":
    target_width = int(round(custom_width / 8) * 8)
    target_height = int(round(custom_height / 8) * 8)
    dimension_label = f"Custom ({target_width}×{target_height})"
else:
    ASPECT_RATIO_SPECS = {
        "1:1 (Square)": (1, 1),
        "16:9 (Widescreen)": (16, 9),
        "9:16 (Portrait Widescreen)": (9, 16),
        "21:9 (Ultrawide)": (21, 9),
        "9:21 (Tall Ultrawide)": (9, 21),
        "4:3 (Standard)": (4, 3),
        "3:4 (Portrait Standard)": (3, 4),
        "3:2 (Photo)": (3, 2),
        "2:3 (Portrait Photo)": (2, 3),
        "5:4 (Landscape Tall)": (5, 4),
        "4:5 (Portrait Tall)": (4, 5),
        "1:2 (Tall Vertical)": (1, 2),
        "2:1 (Panoramic)": (2, 1)
    }
    w_ratio, h_ratio = ASPECT_RATIO_SPECS.get(aspect_ratio, (1, 1))
    total_pixels = megapixels * 1024 * 1024
    scale = math.sqrt(total_pixels / (w_ratio * h_ratio))
    target_width = int(round(w_ratio * scale / 8) * 8)
    target_height = int(round(h_ratio * scale / 8) * 8)
    dimension_label = f"{aspect_ratio} ({target_width}×{target_height})"

# Adaptive encoder scale for high-res canvases
pixel_count = target_width * target_height
encoder_resolution = 2048 if (pixel_count >= 1536 * 1024 or max(target_width, target_height) >= 1536) else 1024

# Workflow Definition
workflow = {
  "461": {
    "inputs": {
      "filename_prefix": "Qwen_image_2.1",
      "format": "png",
      "format.bit_depth": "8-bit",
      "format.input_color_space": "sRGB",
      "images": ["459:457", 0]
    },
    "class_type": "SaveImageAdvanced",
    "_meta": {"title": "Save Image (Advanced)"}
  },
  "459:451": {
    "inputs": {
      "unet_name": "qwen_image_2.1_int8_convrot.safetensors",
      "weight_dtype": "default"
    },
    "class_type": "UNETLoader",
    "_meta": {"title": "Load Diffusion Model"}
  },
  "459:452": {
    "inputs": {
      "prompt": prompt_text,
      "negative_prompt": negative_prompt,
      "resolution": encoder_resolution,
      "clip": ["459:453", 0]
    },
    "class_type": "TextEncodeQwenImage21",
    "_meta": {"title": "Text Encode Qwen Image 2.1"}
  },
  "459:453": {
    "inputs": {
      "clip_name": "qwen3vl_8b_int8_convrot.safetensors",
      "type": "qwen_image",
      "device": "default"
    },
    "class_type": "CLIPLoader",
    "_meta": {"title": "Load CLIP"}
  },
  "459:454": {
    "inputs": {
      "vae_name": "qwen_image_2.1_vae_bf16.safetensors"
    },
    "class_type": "VAELoader",
    "_meta": {"title": "Load VAE"}
  },
  "459:456": {
    "inputs": {
      "width": target_width,
      "height": target_height,
      "batch_size": 4
    },
    "class_type": "EmptyLatentImage",
    "_meta": {"title": "Empty Latent Image"}
  },
  "459:457": {
    "inputs": {
      "samples": ["459:458", 0],
      "vae": ["459:454", 0]
    },
    "class_type": "VAEDecode",
    "_meta": {"title": "VAE Decode"}
  },
  "459:458": {
    "inputs": {
      "seed": seed,
      "steps": steps,
      "cfg": 1,
      "sampler_name": "euler",
      "scheduler": "simple",
      "denoise": 1,
      "model": ["459:480", 0],
      "positive": ["459:452", 0],
      "negative": ["459:452", 1],
      "latent_image": ["459:456", 0]
    },
    "class_type": "KSampler",
    "_meta": {"title": "KSampler"}
  },
  "459:480": {
    "inputs": {
      "device": "auto",
      "dtype": "default",
      "model": ["459:451", 0]
    },
    "class_type": "QwenImage21Cache",
    "_meta": {"title": "Qwen Image 2.1 Cache"}
  }
}

# Dynamic LoRA Routing
if use_lora:
    workflow["lora_loader"] = {
        "inputs": {
            "lora_name": lora_name,
            "strength_model": lora_model_strength,
            "strength_clip": lora_clip_strength,
            "model": ["459:451", 0],
            "clip": ["459:453", 0]
        },
        "class_type": "LoraLoader",
        "_meta": {"title": "LoRA Loader"}
    }
    workflow["459:480"]["inputs"]["model"] = ["lora_loader", 0]
    workflow["459:452"]["inputs"]["clip"] = ["lora_loader", 1]
else:
    workflow["459:480"]["inputs"]["model"] = ["459:451", 0]
    workflow["459:452"]["inputs"]["clip"] = ["459:453", 0]

# Server Guard & Auto-Revive
server_address = "127.0.0.1:8188"
client_id = str(uuid.uuid4())

def purge_backend_memory():
    """Flushes PyTorch CUDA allocation cache without dropping base weights."""
    try:
        payload = json.dumps({"unload_models": False, "free_memory": True}).encode('utf-8')
        req = urllib.request.Request(f"http://{server_address}/free", data=payload, headers={'Content-Type': 'application/json'})
        with urllib.request.urlopen(req, timeout=5) as resp:
            pass
    except Exception:
        pass

def ensure_server_alive():
    try:
        r = requests.get(f"http://{server_address}/system_stats", timeout=1)
        if r.status_code == 200:
            return True
    except requests.exceptions.RequestException:
        pass

    print("Backend server was offline. Rebooting server...")
    if 'launch_inference_server' in globals():
        launch_inference_server()
    else:
        raise RuntimeError("Server process offline. Please run Cell 3 to launch the server.")

    t_wait = time.time()
    while time.time() - t_wait < 60:
        try:
            r = requests.get(f"http://{server_address}/system_stats", timeout=1)
            if r.status_code == 200:
                print("✓ Backend server recovered and ready.")
                return True
        except requests.exceptions.RequestException:
            pass
        time.sleep(1.2)
    raise RuntimeError("Failed to connect to backend server. Port 8188 refused connection.")

ensure_server_alive()
# Pre-emptive memory cleanup before queuing generation
purge_backend_memory()

# Real-Time Dashboard Renderer
display_handle = display(HTML(""), display_id="dashboard_ui")

def render_dashboard(stage: str, node_title: str, progress_pct: int, elapsed_sec: float, done: bool = False, error: bool = False):
    if error:
        badge_color = "#f85149"
        status_text = "ERROR"
        grad = "#da3633"
    elif done:
        badge_color = "#3fb950"
        status_text = "COMPLETED"
        grad = "#2ea043"
    else:
        badge_color = "#58a6ff"
        status_text = "PROCESSING"
        grad = "linear-gradient(90deg, #1f6feb, #58a6ff)"

    bar_fill = min(100, max(0, progress_pct))

    html = f"""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; background: #0d1117; color: #e6edf3; border: 1px solid #30363d; border-radius: 10px; padding: 18px; margin: 12px 0; max-width: 780px; box-shadow: 0 8px 24px rgba(0,0,0,0.3);">
      <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 14px; border-bottom: 1px solid #21262d; padding-bottom: 10px;">
        <span style="font-size: 13px; font-weight: 600; letter-spacing: 0.5px; color: #58a6ff;">✦ QWEN-IMAGE-2.1 INFERENCE ENGINE</span>
        <span style="background: rgba(88, 166, 255, 0.12); color: {badge_color}; border: 1px solid {badge_color}40; padding: 3px 10px; border-radius: 20px; font-size: 11px; font-weight: 600;">{status_text}</span>
      </div>

      <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 10px; font-size: 12px; margin-bottom: 14px;">
        <div style="background: #161b22; padding: 8px 12px; border-radius: 6px; border: 1px solid #21262d;">
          <div style="color: #8b949e; font-size: 10px; text-transform: uppercase;">Seed</div>
          <div style="color: #f0f6fc; font-weight: 600; font-family: monospace; margin-top: 2px;">{seed}</div>
        </div>
        <div style="background: #161b22; padding: 8px 12px; border-radius: 6px; border: 1px solid #21262d;">
          <div style="color: #8b949e; font-size: 10px; text-transform: uppercase;">Dimensions</div>
          <div style="color: #f0f6fc; font-weight: 600; margin-top: 2px;">{target_width} × {target_height}</div>
        </div>
        <div style="background: #161b22; padding: 8px 12px; border-radius: 6px; border: 1px solid #21262d;">
          <div style="color: #8b949e; font-size: 10px; text-transform: uppercase;">LoRA Status</div>
          <div style="color: {'#3fb950' if use_lora else '#8b949e'}; font-weight: 600; margin-top: 2px;">{'Active' if use_lora else 'Disabled'}</div>
        </div>
        <div style="background: #161b22; padding: 8px 12px; border-radius: 6px; border: 1px solid #21262d;">
          <div style="color: #8b949e; font-size: 10px; text-transform: uppercase;">Elapsed Time</div>
          <div style="color: #58a6ff; font-weight: 600; font-family: monospace; margin-top: 2px;">{elapsed_sec:.1f}s</div>
        </div>
      </div>

      <div style="margin-bottom: 6px; display: flex; justify-content: space-between; font-size: 11px;">
        <span style="color: #8b949e;">Current Pipeline Stage: <strong style="color: #f0f6fc;">{stage}</strong></span>
        <span style="color: #58a6ff; font-family: monospace;">{bar_fill}%</span>
      </div>
      <div style="height: 6px; background: #21262d; border-radius: 3px; overflow: hidden;">
        <div style="height: 100%; width: {bar_fill}%; background: {grad}; transition: width 0.15s ease-out;"></div>
      </div>

      <div style="margin-top: 10px; font-size: 11px; color: #8b949e; font-family: monospace;">
        ⚡ Active Execution Unit: <span style="color: #79c0ff;">{node_title}</span>
      </div>
    </div>
    """
    display_handle.update(HTML(html))

# Strict Queue Submission with Error Surface
def queue_prompt(prompt_workflow):
    payload = json.dumps({"prompt": prompt_workflow, "client_id": client_id}).encode('utf-8')
    req = urllib.request.Request(f"http://{server_address}/prompt", data=payload, headers={'Content-Type': 'application/json'})
    try:
        with urllib.request.urlopen(req) as resp:
            res_data = json.loads(resp.read().decode('utf-8'))
            if 'error' in res_data:
                raise ValueError(f"Prompt validation error: {res_data['error']}")
            if 'node_errors' in res_data and res_data['node_errors']:
                raise ValueError(f"Node error in workflow: {res_data['node_errors']}")
            if 'prompt_id' not in res_data:
                raise ValueError(f"Server rejected prompt without ID: {res_data}")
            return res_data
    except urllib.error.HTTPError as e:
        err_body = e.read().decode('utf-8')
        raise ValueError(f"Server rejected prompt (HTTP {e.code}): {err_body}")

def get_image(filename, subfolder, folder_type):
    params = urllib.parse.urlencode({"filename": filename, "subfolder": subfolder, "type": folder_type})
    with urllib.request.urlopen(f"http://{server_address}/view?{params}") as response:
        return response.read()

start_t = time.time()
render_dashboard("Validating Dimensions & Queueing...", f"Target: {target_width}×{target_height}", 0, 0.0)

prompt_res = queue_prompt(workflow)
prompt_id = prompt_res['prompt_id']

ws = websocket.WebSocket()
ws.connect(f"ws://{server_address}/ws?clientId={client_id}")
ws.settimeout(1.0)

current_pct = 5

try:
    while True:
        if 'server_proc' in globals() and server_proc.poll() is not None:
            render_dashboard("Server Terminated", "Process Crashed", current_pct, time.time() - start_t, error=True)
            raise RuntimeError("Backend server process terminated unexpectedly. Check cell logs above.")

        try:
            out = ws.recv()
        except websocket.WebSocketTimeoutException:
            render_dashboard(f"Diffusion Step Computation ({target_width}×{target_height})...", "In Progress", current_pct, time.time() - start_t)
            continue

        if isinstance(out, str):
            msg = json.loads(out)
            msg_type = msg.get('type')
            elapsed = time.time() - start_t

            if msg_type == 'execution_error':
                err_info = msg.get('data', {})
                err_msg = err_info.get('exception_message', 'Execution error')
                render_dashboard(f"Failed in {err_info.get('node_type')}", err_msg, current_pct, elapsed, error=True)
                raise RuntimeError(f"Pipeline Execution Error: {err_msg}")

            if msg_type == 'execution_interrupted':
                render_dashboard("Pipeline Canceled", "Interrupted", current_pct, elapsed, error=True)
                break

            if msg_type == 'executing':
                node_id = msg['data'].get('node')
                msg_prompt_id = msg['data'].get('prompt_id')
                if node_id is None:
                    if msg_prompt_id is None or msg_prompt_id == prompt_id:
                        render_dashboard(f"Completed ({target_width}×{target_height})", "Finalizing Latent Decode", 100, elapsed, done=True)
                        break
                elif str(node_id) in workflow:
                    node_name = workflow[str(node_id)].get('_meta', {}).get('title', f"Node {node_id}")
                    render_dashboard(f"Pipeline Dispatch ({target_width}×{target_height})", node_name, current_pct, elapsed)

            elif msg_type == 'progress':
                data = msg['data']
                step = data['value']
                total_steps = data['max']
                current_pct = 10 + int((step / total_steps) * 85)
                render_dashboard(f"Sampling {step}/{total_steps} ({target_width}×{target_height})", "KSampler (Euler Simple)", current_pct, elapsed)
finally:
    try:
        ws.close()
    except Exception:
        pass

# Retrieve Generated Output
history_req = urllib.request.urlopen(f"http://{server_address}/history/{prompt_id}")
history = json.loads(history_req.read())[prompt_id]

output_images = []
for node_id in history['outputs']:
    node_output = history['outputs'][node_id]
    if 'images' in node_output:
        for img_info in node_output['images']:
            raw_data = get_image(img_info['filename'], img_info['subfolder'], img_info['type'])
            output_images.append(Image.open(io.BytesIO(raw_data)))

# Post-generation memory flush (ensures server is clean for next run)
purge_backend_memory()

# Render Finished Gallery
for img in output_images:
    display(HTML(f"""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; background: #0d1117; border: 1px solid #30363d; border-radius: 8px; padding: 12px; margin-top: 10px; max-width: 780px;">
      <div style="color: #8b949e; font-size: 11px; margin-bottom: 8px;">Resolution: <strong style="color: #f0f6fc;">{img.width} × {img.height}</strong> | Render Time: <strong style="color: #58a6ff;">{time.time() - start_t:.2f}s</strong></div>
    </div>
    """))
    display(img)
    #save image to google drive
    save_dir = '/content/drive/MyDrive'
    Path(save_dir).mkdir(parents=True, exist_ok=True)
    img.save('/content/drive/MyDrive/image.png')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')